# Subset the earliest_engagement data to engagement before ages 4 and 7


In [ ]:
import pandas as pd
import zipfile
import ast
import re
import json
import numpy as np
import statistics
import datetime
import matplotlib.pyplot as plt
import gzip
import os
import math

In [ ]:
earliest = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/earliest.csv.zip')
# put in all of the AFC data
with gzip.open('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz') as f:
    patient_baseline = pd.read_csv(f, low_memory = False)
  


In [ ]:
# require patients to have interacted with AFC before age 7
baseline_engaged = pd.merge(patient_baseline, earliest, left_on = 'patientuid', right_on = 'person_source_value')
valid_date = [x[0] == '1' or x[0] == '2' for x in baseline_engaged['date']]
baseline_engaged = baseline_engaged[valid_date]

baseline_engaged.loc[:,'earliest_engaged'] = pd.to_datetime(baseline_engaged['date'], format='mixed')
baseline_engaged.loc[:,'earliest_ordinal'] = [x.toordinal() for x in baseline_engaged['earliest_engaged']]

baseline_engaged.loc[:,'dob'] = pd.to_datetime(baseline_engaged['dob'], format='mixed')
baseline_engaged.loc[:,'dob_ordinal'] = [x.toordinal() for x in baseline_engaged['dob']]
baseline_engaged = baseline_engaged[['patientuid', 'earliest_engaged', 'earliest_ordinal', 'dob', 'dob_ordinal']]

baseline_engaged.loc[:, 'years_to_engage'] = (baseline_engaged['earliest_ordinal']-baseline_engaged['dob_ordinal'])/365.0
baseline_engaged.loc[:,'len_history_years'] = (datetime.datetime.today().toordinal() - baseline_engaged['earliest_ordinal'])/365.0
baseline_engaged.loc[:,'age_years'] = (datetime.datetime.today().toordinal() - baseline_engaged['dob_ordinal'])/365.0

baseline_engaged = baseline_engaged[baseline_engaged['years_to_engage'] >= 0]

In [ ]:
baseline_engaged_7 = baseline_engaged[baseline_engaged['years_to_engage'] <= 7]
baseline_engaged_4 = baseline_engaged[baseline_engaged['years_to_engage'] <= 4]



In [ ]:
len(baseline_engaged_7), len(baseline_engaged_4)

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
# save the patientuids of patients who engaged before 7, 4
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/engaged_before_7.csv', baseline_engaged_7['patientuid'])
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/engaged_before_4.csv', baseline_engaged_4['patientuid'])

In [ ]:
baseline_engaged_filtered = baseline_engaged[['patientuid', 'years_to_engage']]

In [ ]:
len(baseline_engaged_filtered)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/earliest_engaged.csv', baseline_engaged_filtered)